# Item 22 — O mapa de onde a estrutura paga

Exigido pelo caso 3 do [ADR-0001](../docs/adr/0001-precompromisso-de-falseamento.md): a proposta não
supera o baseline sem grafo nem o grafo embaralhado na média, e a dissertação passa a entregar **onde**
a estrutura paga. Spec: [`phase-6b-mapa-estrutura`](../.specs/features/phase-6b-mapa-estrutura/spec.md).

Este notebook **não treina nada e não usa GPU**. Lê as predições do item 10 e o `timeseries.parquet`,
estratifica as origens do Modo 2 e testa, por estrato, a proposta contra os dois controles.

## O que foi fixado antes de qualquer número estratificado

**Hipótese a priori.** A estrutura paga em **estreia** e **reentrada**: sem histórico próprio na
semana de origem, a vizinhança é a única informação nova, e a GNN é ancorada na persistência, que
é cega ali. Não paga em **maduro_mono**: a persistência já resume tudo. Sem previsão para `jovem` e
`maduro_multi`. O notebook compara o veredito com esta hipótese ao final, sem reescrevê-la.

**Estratos** (mutuamente exclusivos, só com semanas `≤` origem):

| Estrato | Definição na semana de origem `w` |
|---|---|
| `estreia` | nenhuma semana on-chart em `≤ w` |
| `reentrada` | `w` fora do chart, com ao menos uma semana on-chart antes |
| `jovem` | `w` on-chart e `w − primeira semana on-chart ≤ 8` |
| `maduro_mono` | `w` on-chart, idade `> 8`, uma onda até `w` |
| `maduro_multi` | `w` on-chart, idade `> 8`, duas ou mais ondas até `w` |

Onda nova = ao menos 4 semanas seguidas fora do chart. Origem sem série em `timeseries.parquet` é
`sem_serie` e fica fora dos testes.

**Limiares.** Célula com menos de 30 músicas é `insuficiente` (fora do teste e do Holm). IC bootstrap 95%
(10.000 reamostras, semente 42) da diferença de erro quadrático **por música**; `vence` = IC inteiro abaixo
de zero e `p` Holm < 0,05; `equivalente` = IC dentro de ±5% do erro quadrático médio do controle.
Holm: uma família por split, os dois contrastes juntos (no máximo 30 células). **Estrutura paga** =
vence os dois controles no mesmo horizonte em ≥ 2 de 3 horizontes, nos dois splits.

## 0. Ambiente

No Colab, clona o repositório. As predições do item 10 (`results/comparacao_final/`) precisam estar
**commitadas e empurradas**; se não estiverem, copie `comparacao_final/` para
`MyDrive/music-influence-gnn/` e o notebook as procura lá. Não precisa de GPU nem de PyG.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone antigo ficaria para trás em silêncio.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "tabulate"], check=True)
    print("✓ Colab pronto. cwd =", os.getcwd())
else:
    print("Local — nada a clonar.")

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

from music_diffusion_gnn.evaluation.longhits import onchart_weeks
from music_diffusion_gnn.evaluation.mapa import (
    build_map, by_chart_rmse, margin_sensitivity, sensitivity_by_seed, structure_pays,
)
from music_diffusion_gnn.evaluation.strata import STRATA, assign_strata, strata_counts

print("ROOT =", ROOT)

In [ ]:
# ---- parâmetros fixados na spec (não mexer depois de ver o resultado) ----
YOUNG_WEEKS = 8
WAVE_GAP    = 4
MIN_SONGS   = 30
EQUIV_FRAC  = 0.05
B           = 10_000
SEED        = 42
TRATADO     = "proposta"
CONTROLES   = ("sem_grafo", "embaralhado")

DRIVE = Path("/content/drive/MyDrive/music-influence-gnn")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

candidatos = [ROOT / "results" / "comparacao_final" / "predicoes_cinco_modelos.parquet",
              DRIVE / "comparacao_final" / "predicoes_cinco_modelos.parquet"]
PRED_PATH = next((p for p in candidatos if p.exists()), None)
if PRED_PATH is None:
    raise FileNotFoundError(
        "predicoes_cinco_modelos.parquet não encontrado. Procurei em:\n  "
        + "\n  ".join(map(str, candidatos))
        + "\nCommite e empurre results/comparacao_final/ ou copie a pasta para o Drive.")

TS_PATH = ROOT / "data" / "processed" / "timeseries.parquet"
OUT = ROOT / "results" / "mapa_estrutura"
OUT.mkdir(parents=True, exist_ok=True)
print("predições ←", PRED_PATH)
print("saída     →", OUT)

## 1. Dados e estratos

In [ ]:
ts = pd.read_parquet(TS_PATH)
oc = onchart_weeks(ts)
series_keys = set(map(tuple, ts[["song_id", "chart"]].drop_duplicates().itertuples(index=False, name=None)))
del ts

preds = pd.read_parquet(PRED_PATH)
preds = preds[preds["modelo"].isin([TRATADO, *CONTROLES])].copy()

origens = preds[["song_id", "chart", "origin_week"]].drop_duplicates().reset_index(drop=True)
origens["estrato"] = assign_strata(origens, oc, young_weeks=YOUNG_WEEKS, wave_gap=WAVE_GAP,
                                   series_keys=series_keys)
print(f"{len(origens):,} origens distintas;", f"{len(series_keys):,} séries (música, chart)")
display(origens.estrato.value_counts().reindex([*STRATA, "sem_serie"]).fillna(0).astype(int).to_frame("origens"))

In [ ]:
# Contagens por split × estrato × horizonte, sobre as origens de UM modelo.
# 'todas' = todas as origens do Modo 2; 'onchart' = as que entram nos testes (alvo no chart).
base = preds[preds.modelo == TRATADO].merge(origens, on=["song_id", "chart", "origin_week"])
cont_todas   = strata_counts(base, min_songs=MIN_SONGS)
cont_onchart = strata_counts(base[base.onchart], min_songs=MIN_SONGS)

print("linhas de origem por split:", base.groupby("split_regime").size().to_dict(),
      "| o item 10 reporta 141.504 (current) e 53.982 (pre_pandemia) por modelo")
assert cont_todas.groupby("split_regime").n_origens.sum().to_dict() == base.groupby("split_regime").size().to_dict()

print("\n— todas as origens —")
display(cont_todas.pivot_table(index=["split_regime", "estrato"], columns="k", values="n_musicas"))
print("— origens com alvo on-chart (amostra dos testes); insuficiente = menos de 30 músicas —")
display(cont_onchart.pivot_table(index=["split_regime", "estrato"], columns="k", values="n_musicas"))
display(cont_onchart[cont_onchart.insuficiente][["split_regime", "estrato", "k", "n_musicas"]])

## 2. O mapa

Uma linha por (split, estrato, horizonte, controle). `mean_diff < 0` = a proposta erra menos.
A leitura é **on-chart** (ADR-0004). Roda 10.000 reamostras por célula testada; leva alguns minutos.

In [ ]:
mapa = build_map(preds, origens, treated=TRATADO, controls=CONTROLES,
                 B=B, seed=SEED, min_songs=MIN_SONGS, equiv_frac=EQUIV_FRAC)
mapa.to_parquet(OUT / "mapa.parquet", index=False)

cols = ["split_regime", "estrato", "k", "contraste", "n_musicas", "mean_diff", "ci_lo", "ci_hi",
        "p_holm", "win_rate", "rotulo"]
display(mapa[cols].round(5))
print("\nrótulos:"); display(mapa.groupby(["split_regime", "contraste"]).rotulo.value_counts().unstack(fill_value=0))

In [ ]:
paga = structure_pays(mapa, controls=CONTROLES)
paga.to_parquet(OUT / "estrutura_paga.parquet", index=False)
display(paga)

In [ ]:
# O veredito contra a hipótese escrita no topo do notebook (sem reescrevê-la).
HIPOTESE = {"estreia": "estrutura paga", "reentrada": "estrutura paga", "maduro_mono": "nao paga"}
v = paga.set_index("estrato")["veredito"]
linhas = []
for estrato, previsto in HIPOTESE.items():
    obtido = v.get(estrato, "ausente")
    if obtido == "nao testavel":
        desfecho = "inconclusivo (não testável)"
    elif obtido == previsto:
        desfecho = "confirmada"
    else:
        desfecho = "refutada"
    linhas.append({"estrato": estrato, "previsto": previsto, "obtido": obtido, "hipotese": desfecho})
hip = pd.DataFrame(linhas)
hip.to_parquet(OUT / "hipotese_vs_resultado.parquet", index=False)
display(hip)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
cores = {"sem_grafo": "#6C5CE0", "embaralhado": "#E4572E"}
for ax, (split, g) in zip(axes, mapa[~mapa.insuficiente].groupby("split_regime")):
    g = g.assign(rel=lambda d: d.mean_diff / d.mean_b * 100,
                 lo=lambda d: d.ci_lo / d.mean_b * 100, hi=lambda d: d.ci_hi / d.mean_b * 100)
    estratos = [e for e in STRATA if e in set(g.estrato)]
    for j, (contraste, gc) in enumerate(g.groupby("contraste")):
        for i, e in enumerate(estratos):
            ge = gc[gc.estrato == e].sort_values("k")
            x = i + (j - 0.5) * 0.35 + np.linspace(-0.1, 0.1, len(ge))
            ax.errorbar(x, ge.rel, yerr=[ge.rel - ge.lo, ge.hi - ge.rel], fmt="o", ms=4,
                        color=cores[contraste], label=contraste if i == 0 else None, capsize=2)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(len(estratos)), estratos, rotation=20)
    ax.set_title(split); ax.set_ylabel("Δ erro quadrático vs controle (%) — abaixo de 0: proposta melhor")
axes[0].legend(title="controle")
fig.tight_layout(); plt.show()

## 3. Sensibilidade

Três perguntas: o rótulo resiste à margem de equivalência? Muda por chart? Repete seed a seed?
A primeira e a segunda são locais; a terceira precisa dos `mode2_*_completo.parquet` do item 09 no Drive.

In [ ]:
mudam = margin_sensitivity(mapa, fracs=(0.025, 0.10))
mudam.to_parquet(OUT / "sensibilidade_margem.parquet", index=False)
print(f"{len(mudam)} células mudam de rótulo com margem de 2,5% ou 10% (de {int((~mapa.insuficiente).sum())} testadas)")
display(mudam)

por_chart = by_chart_rmse(preds, origens, treated=TRATADO, controls=CONTROLES)
por_chart.to_parquet(OUT / "diferenca_por_chart.parquet", index=False)
print("\nDiferença de RMSE por chart (descritiva, sem teste; negativo = proposta melhor):")
display(por_chart.pivot_table(index=["split_regime", "estrato", "contraste"], columns=["chart", "k"],
                              values="dif_rmse").round(4))

In [ ]:
MATRIZ = DRIVE / "avaliacao_matriz" if IS_COLAB else ROOT / "results" / "avaliacao_matriz"
arquivos = sorted(MATRIZ.glob("mode2_*_completo.parquet"))
if not arquivos:
    print(f"sem mode2_*_completo.parquet em {MATRIZ}: a concordância por seed fica para quando eles estiverem à mão.")
    por_seed = None
else:
    seeds = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)
    seeds = seeds.rename(columns={"variante": "modelo"})
    seeds = seeds[seeds["modelo"].isin([TRATADO, *CONTROLES])]
    por_seed = sensitivity_by_seed(seeds, origens, mapa, treated=TRATADO, controls=CONTROLES,
                                   B=B, seed=SEED, min_songs=MIN_SONGS, equiv_frac=EQUIV_FRAC)
    por_seed.to_parquet(OUT / "sensibilidade_seed.parquet", index=False)
    testadas = por_seed[por_seed.rotulo != "insuficiente"]
    print("células testadas por nº de seeds que repetem o rótulo principal:")
    display(testadas.seeds_concordam.value_counts().sort_index().to_frame("células"))
    display(testadas[testadas.seeds_concordam < testadas.n_seeds])

## 4. Relatório

In [ ]:
def tabela_md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

linhas = ["# O mapa de onde a estrutura paga", "",
          "Gerado por `notebooks/item22_mapa_estrutura_colab.ipynb`. A hipótese e os limiares abaixo foram "
          "fixados na spec antes de qualquer número estratificado.", "",
          f"Parâmetros: jovem ≤ {YOUNG_WEEKS} semanas, nova onda após {WAVE_GAP} semanas fora do chart, "
          f"mínimo {MIN_SONGS} músicas, margem de equivalência {EQUIV_FRAC:.0%}, bootstrap {B:,} (semente {SEED}).", "",
          "## Hipótese contra o resultado", "", tabela_md(hip), "",
          "## Veredito por estrato", "", tabela_md(paga), "",
          "## Contagens (alvo on-chart)", "", tabela_md(cont_onchart), "",
          "## Mapa", "", tabela_md(mapa[cols].round(5)), "",
          "## Células que mudam com a margem de equivalência", "", tabela_md(mudam.round(5)), ""]
if por_seed is not None:
    linhas += ["## Concordância entre seeds", "",
               tabela_md(por_seed[por_seed.rotulo != "insuficiente"].round(5)), ""]
(OUT / "relatorio.md").write_text("\n".join(linhas), encoding="utf-8")

if IS_COLAB:
    destino = DRIVE / "mapa_estrutura"
    shutil.copytree(OUT, destino, dirs_exist_ok=True)
    print("cópia no Drive →", destino)
print("escrito:", sorted(p.name for p in OUT.iterdir()))
print("\nPara a dissertação: copie relatorio.md para docs/mapa-onde-a-estrutura-paga.md e comite os parquets.")